# Azure AI Translator: Multilingual Text Translation

In this notebook you will learn how to use **Azure AI Translator** to:

1. Translate text from one language to another
2. Translate into **multiple** target languages in a single call
3. Use **automatic language detection** when the source language is unknown

---

### Prerequisites: Deploy the Azure Resource

Before running this notebook, you need an **Azure AI Translator** resource. Instead of configuring it manually in the Azure Portal, deploy the included **ARM template** which creates the resource in the correct configuration automatically.

> **Tip:** Open `deploy-translator.parameters.json` to customize the resource name, region, or SKU before deploying. The default SKU is **S1** (paid). Change it to **F0** for the free tier.

**1. Deploy the resource** — run the cell below. Set `RESOURCE_GROUP` in `.env` if your resource group is not named `Foundry`.

In [ ]:
import os
import subprocess
import shutil
from pathlib import Path


def read_env_value(name: str, default: str | None = None) -> str | None:
    value = os.getenv(name)
    if value:
        return value

    env_path = Path(".env")
    if env_path.exists():
        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, raw_value = line.split("=", 1)
            if key.strip() == name:
                return raw_value.strip().strip('"').strip("'")

    return default


RESOURCE_GROUP = read_env_value("RESOURCE_GROUP", "Foundry")

# Resolve the full path to the Azure CLI (needed for Windows where az is a .cmd file)
AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

result = subprocess.run(
    [AZ, "deployment", "group", "create",
     "--resource-group", RESOURCE_GROUP,
     "--name", "deploy-translator",
     "--template-file", "deploy-translator.json",
     "--parameters", "@deploy-translator.parameters.json"],
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

**2. Assign the required RBAC role** so your identity can use the resource. Set `RESOURCE_GROUP` in `.env` if your resource group is not named `Foundry`.

> **Note:** RBAC role assignments can take **1–2 minutes** to propagate. If you get a 401 error on your first run, wait a moment and try again.

In [ ]:
import os
import subprocess
import shutil
from pathlib import Path


def read_env_value(name: str, default: str | None = None) -> str | None:
    value = os.getenv(name)
    if value:
        return value

    env_path = Path(".env")
    if env_path.exists():
        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, raw_value = line.split("=", 1)
            if key.strip() == name:
                return raw_value.strip().strip('"').strip("'")

    return default


RESOURCE_GROUP = read_env_value("RESOURCE_GROUP", "Foundry")

AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

# Step A: Get the signed-in user's Object ID
user_result = subprocess.run(
    [AZ, "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"],
    capture_output=True, text=True, encoding="utf-8"
)
user_id = user_result.stdout.strip()
print(f"Your User ID: {user_id}")

# Step B: Get the resource ID from the deployment output
scope_result = subprocess.run(
    [AZ, "deployment", "group", "show",
     "--resource-group", RESOURCE_GROUP,
     "--name", "deploy-translator",
     "--query", "properties.outputs.resourceId.value", "-o", "tsv"],
    capture_output=True, text=True, encoding="utf-8"
)
resource_id = scope_result.stdout.strip()
print(f"Resource ID:  {resource_id}")

# Step C: Assign the RBAC role
role_result = subprocess.run(
    [AZ, "role", "assignment", "create",
     "--assignee", user_id,
     "--role", "Cognitive Services User",
     "--scope", resource_id],
    capture_output=True, text=True, encoding="utf-8"
)
print(role_result.stdout)
if role_result.returncode != 0:
    print("ERROR:", role_result.stderr)

**3. Update the `.env` file** in this folder with the values from the deployment output:

```
TRANSLATOR_ENDPOINT="https://<your-account-name>.cognitiveservices.azure.com/"
TRANSLATOR_REGION="eastus2"
```

You can retrieve these values by running the cell below. Set `RESOURCE_GROUP` in `.env` if your resource group is not named `Foundry`.

In [ ]:
import os
import subprocess
import shutil
from pathlib import Path


def read_env_value(name: str, default: str | None = None) -> str | None:
    value = os.getenv(name)
    if value:
        return value

    env_path = Path(".env")
    if env_path.exists():
        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, raw_value = line.split("=", 1)
            if key.strip() == name:
                return raw_value.strip().strip('"').strip("'")

    return default


RESOURCE_GROUP = read_env_value("RESOURCE_GROUP", "Foundry")

AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

result = subprocess.run(
    [AZ, "deployment", "group", "show",
     "--resource-group", RESOURCE_GROUP,
     "--name", "deploy-translator",
     "--query", "properties.outputs"],
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

### Authentication

This notebook uses **Microsoft Entra ID** (`DefaultAzureCredential`) instead of API keys.
Make sure you are logged in with `az login`.

> **Note:** The Translator SDK's `TranslatorCredential` requires an API key.
> When using Entra ID, we obtain a token manually and pass it via the
> REST API directly using `requests`.

## Step 1: Install Dependencies

In [ ]:
%pip install azure-identity requests python-dotenv

## Step 2: Load Environment Variables and Obtain a Token

The Translator Python SDK (`azure-ai-translation-text`) currently requires
an API key via `TranslatorCredential`. Since we want key-free auth, we'll
call the Translator REST API directly with an Entra ID bearer token.

In [ ]:
import os
import requests
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

load_dotenv(override=True)

translator_endpoint = os.getenv("TRANSLATOR_ENDPOINT")   # custom domain endpoint
translator_region = os.getenv("TRANSLATOR_REGION")

# Obtain a bearer token from Entra ID
credential = DefaultAzureCredential()
token = credential.get_token("https://cognitiveservices.azure.com/.default").token

# Common headers for every Translator REST call
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
    "Ocp-Apim-Subscription-Region": translator_region,
}

print("Endpoint:", translator_endpoint)
print("Region:", translator_region)
print("Token obtained:", "Yes" if token else "No")

## Step 3: Translate Text to a Single Language

We translate an English sentence into French using the `/translate` endpoint.

In [ ]:
# Build the request URL
translate_url = f"{translator_endpoint}translator/text/v3.0/translate"

# Source text in English
original_text = "Artificial intelligence is transforming every industry around the world."

response = requests.post(
    translate_url,
    headers=headers,
    params={"from": "en", "to": "fr", "api-version": "3.0"},
    json=[{"Text": original_text}],
)
response.raise_for_status()
result = response.json()

translated = result[0]["translations"][0]["text"]
print(f"Original (en):    {original_text}")
print(f"Translated (fr):  {translated}")

## Step 4: Translate to Multiple Languages at Once

You can specify several target languages in the `to` parameter and get
all translations back in a single API call.

In [ ]:
source_sentence = "The weather is beautiful today and I feel like going for a walk."

multi_response = requests.post(
    translate_url,
    headers=headers,
    params={"from": "en", "to": ["es", "de", "ja", "pt"], "api-version": "3.0"},
    json=[{"Text": source_sentence}],
)
multi_response.raise_for_status()
multi_result = multi_response.json()

print(f"Original (en):  {source_sentence}\n")
for translation in multi_result[0]["translations"]:
    print(f"  [{translation['to']}]  {translation['text']}")

## Step 5: Auto-Detect Source Language

If you don't know the source language, simply omit the `from` parameter
and the service will detect it automatically.

In [ ]:
# A sentence in Italian -- we won't tell the API what language it is
mystery_text = "L'intelligenza artificiale sta rivoluzionando il mondo della tecnologia."

auto_response = requests.post(
    translate_url,
    headers=headers,
    params={"to": "en", "api-version": "3.0"},   # no 'from' -- auto-detect
    json=[{"Text": mystery_text}],
)
auto_response.raise_for_status()
auto_result = auto_response.json()

detected = auto_result[0]["detectedLanguage"]
translated_text = auto_result[0]["translations"][0]["text"]

print(f"Original:          {mystery_text}")
print(f"Detected language: {detected['language']} (confidence: {detected['score']:.0%})")
print(f"English:           {translated_text}")